In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import linear_kernel

In [ ]:
df = pd.read_csv('/dataset/shopee_reviews_purchase.csv')
df

In [ ]:
df['user_id'].value_counts()

In [ ]:
df.columns

In [ ]:
df['cmt_date'] = pd.to_datetime(df['cmt_date'])

In [ ]:
df_sorted = df.sort_values(by = ['user_id', 'cmt_date'])

In [ ]:
df_sorted

In [ ]:
index = df_sorted.groupby(['user_id', 'product_id'])['cmt_date'].idxmax()
df_no_dup = df_sorted.loc[index, ['user_id', 'product_id', 'cmt_date']].sort_values(by=['user_id', 'cmt_date'], ascending=False)
df_no_dup

In [ ]:
list_user = df_no_dup['user_id'].value_counts().index[:3952]
df_purchase = df_no_dup[df_no_dup['user_id'].isin(list_user)]
df_purchase

# product data

In [ ]:
df_prod = pd.read_csv('/dataset/shopee_product.csv')
df_prod

In [ ]:
df_prod.columns

In [ ]:
df_prod['num_sold_time'] = df_prod['num_sold_time']/df_prod['num_sold_time'].max()

In [ ]:
df_prod

# Tf-idf

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(analyzer='word')
tfidf_matrix = tf.fit_transform(df_prod['processed_description'].values)
tfidf_matrix

In [ ]:
cosine_sim = linear_kernel(tfidf_matrix.toarray(), tfidf_matrix.toarray())

In [ ]:
cosine_sim

# Phobert

In [ ]:
import torch
phobert_embedding = torch.load('/content_based_embeddings/shopee_description_full.pt', map_location=torch.device('cpu'))
phobert_embedding

In [ ]:
phobert_embedding = phobert_embedding.numpy()

In [ ]:
phobert_cosine_sim = linear_kernel(phobert_embedding, phobert_embedding)
phobert_cosine_sim

In [ ]:
phobert_cosine_sim = phobert_cosine_sim/phobert_cosine_sim.max()

# BLIP

In [ ]:
blip_embedding = np.load('/content_based_embeddings/blip_description_features.npy')
blip_embedding.shape

In [ ]:
blip_cosine_sim = linear_kernel(blip_embedding, blip_embedding)
blip_cosine_sim

# OpenAI

In [ ]:
ada_embedding = np.load('/content_based_embeddings/ada2_description_features.npy')
ada_embedding.shape

In [ ]:
ada_cosine_sim = linear_kernel(ada_embedding, ada_embedding)
ada_cosine_sim

# Recommend

In [ ]:
id = df_prod['product_id']
indices = pd.Series(df_prod.index, index=df_prod['product_id'])

In [ ]:
id_name_dict = df_prod.set_index('product_id')['product_name'].to_dict()
name_id_dict = df_prod.set_index('product_name')['product_id'].to_dict()


In [ ]:
df_purchase.head(20)

In [ ]:
def recommendations(product_id):
  idx = indices[product_id]
  sim_scores = list(enumerate(ada_cosine_sim[idx]))
  sim_scores = sorted(sim_scores, key=lambda x:x[1], reverse=True)
  sim_scores = sim_scores[1:101]
  df_indices = [i[0] for i in sim_scores]

  return id.iloc[df_indices].values[:10]

In [ ]:
result = recommendations(name_id_dict['[N?I ??A NH?T] S?a r?a m?t Kose Softymo Nh?t B?n 220g'])

In [ ]:
id_name_dict[6312991956]

In [ ]:
for prod_id in result.tolist():
  print(id_name_dict[prod_id])

# Evaluation

In [ ]:
df_purchase

In [ ]:
train_index = [i for i in range(0, 12763, 3)]
val_index = [i for i in range(1, 12763, 3)]
test_index = [i for i in range(2, 12763, 3)]

In [ ]:
df_purchase.iloc[test_index, :]['user_id'].value_counts()

In [ ]:
print(len(train_index), len(val_index), len(test_index))

In [ ]:
df_purchase['user_id'].value_counts()

In [ ]:
def PtopK(y_true, y_pred):
  if y_true in y_pred:
    return 1/len(y_pred)
  else:
    return 0
def RtopK(y_true, y_pred):
  if y_true in y_pred:
    return 1
  else:
    return 0

In [ ]:
def MRR(y_true, y_pred):
    if y_true not in y_pred:
        return 0

    rank = 0
    for index, value in enumerate(y_pred):
        if value == y_true:
            rank = 1 / (index + 1)
            break

    return rank

In [ ]:
def NDCG(y_true, y_pred, k=None):
    if y_true not in y_pred:
        return 0.0

    true_index = y_pred.index(y_true)
    ideal_sorted = sorted(y_pred, reverse=True)
    ideal_index = ideal_sorted.index(y_true)

    dcg = 1.0 / np.log2(true_index + 2)
    idcg = 1.0 / np.log2(ideal_index + 2)

    ndcg = dcg / idcg

    if k is not None:
        ideal_sorted = ideal_sorted[:k]
        y_pred = y_pred[:k]
        true_index = y_pred.index(y_true)
        ideal_index = ideal_sorted.index(y_true)
        dcg = 1.0 / np.log2(true_index + 2)
        idcg = 1.0 / np.log2(ideal_index + 2)
        ndcg = dcg / idcg

    return ndcg


In [ ]:
rtopk_val = []
rtopk_test = []
ptopk_val = []
ptopk_test = []
mrr_val = []
mrr_test = []
ndcg_val = []
ndcg_test = []

for user in df_purchase['user_id'].unique():
  temp = df_purchase[df_purchase['user_id']==user]
  x_val = temp.iloc[2, 1]
  y_val = temp.iloc[1, 1]
  y_val_pred = recommendations(x_val).tolist()
  rtopk_val.append(RtopK(y_val, y_val_pred))
  ptopk_val.append(PtopK(y_val, y_val_pred))
  mrr_val.append(MRR(y_val, y_val_pred))
  ndcg_val.append(NDCG(y_val, y_val_pred, 10))

  x_test = temp.iloc[1, 1]
  y_test = temp.iloc[0, 1]
  y_test_pred = recommendations(x_test).tolist()
  rtopk_test.append(RtopK(y_test, y_test_pred))
  ptopk_test.append(PtopK(y_test, y_test_pred))
  mrr_test.append(MRR(y_test, y_test_pred))
  ndcg_test.append(NDCG(y_test, y_test_pred, 10))


In [ ]:
def calculate_mean(numbers):
    if not numbers:
        return 0.0
    return sum(numbers) / len(numbers)

print('rtopk val', calculate_mean(rtopk_val))
print('rtopk test', calculate_mean(rtopk_test))
print('ptopk val',calculate_mean(ptopk_val))
print('ptopk test', calculate_mean(ptopk_test))
print('mrr val', calculate_mean(mrr_val))
print('mrr test', calculate_mean(mrr_test))
print('ndcg val', calculate_mean(ndcg_val))
print('ndcg test', calculate_mean(ndcg_test))

# Parameter finetunning

In [ ]:
score = [0.1*x for x in range(11)]
score

In [ ]:
def recommendations(product_id, scr_score, embedding):
  idx = indices[product_id]
  sim_scores = list(enumerate(embedding[idx]))
  sim_scores = sorted(sim_scores, key=lambda x:x[1], reverse=True)
  sim_scores = sim_scores[1:101]
  df_indices = [i[0] for i in sim_scores]

  luot_mua = df_prod.loc[df_indices, 'num_sold_time']
  temp = pd.DataFrame(sim_scores, columns = ['index', 'score'])
  temp.index=temp['index']
  temp.drop(['index'], axis=1, inplace=True)
  score = temp['score']*scr_score+luot_mua*(1-scr_score)
  score = score.sort_values(ascending=False)
  df_indices = score.index.values.tolist()

  return id.iloc[df_indices].values[:10]


In [ ]:
from tqdm import tqdm
index = 0
df = pd.DataFrame(columns = ['score_weight', 'popular_weight', 'rtopk val', 'rtopk test', 'ptopk val', 'ptopk test', 'mrr val', 'mrr test', 'ndcg val', 'ndcg test'])
for scr in tqdm(score):
  rtopk_val = []
  rtopk_test = []
  ptopk_val = []
  ptopk_test = []
  mrr_val = []
  mrr_test = []
  ndcg_val = []
  ndcg_test = []

  for user in df_purchase['user_id'].unique():
    temp = df_purchase[df_purchase['user_id']==user]
    x_val = temp.iloc[2, 1]
    y_val = temp.iloc[1, 1]
    y_val_pred = recommendations(x_val, scr, ada_cosine_sim).tolist()
    rtopk_val.append(RtopK(y_val, y_val_pred))
    ptopk_val.append(PtopK(y_val, y_val_pred))
    mrr_val.append(MRR(y_val, y_val_pred))
    ndcg_val.append(NDCG(y_val, y_val_pred, 10))

    x_test = temp.iloc[1, 1]
    y_test = temp.iloc[0, 1]
    y_test_pred = recommendations(x_test, scr, ada_cosine_sim).tolist()
    rtopk_test.append(RtopK(y_test, y_test_pred))
    ptopk_test.append(PtopK(y_test, y_test_pred))
    mrr_test.append(MRR(y_test, y_test_pred))
    ndcg_test.append(NDCG(y_test, y_test_pred, 10))
  dict_temp = {
      'score_weight': scr,
      'popular_weight': 1-scr,
      'rtopk val': calculate_mean(rtopk_val),
      'rtopk test': calculate_mean(rtopk_test),
      'ptopk val': calculate_mean(ptopk_val),
      'ptopk test': calculate_mean(ptopk_test),
      'mrr val': calculate_mean(mrr_val),
      'mrr test': calculate_mean(mrr_test),
      'ndcg val': calculate_mean(ndcg_val),
      'ndcg test': calculate_mean(ndcg_test)
  }
  df = pd.concat([df, pd.DataFrame(dict_temp, index=[index])], ignore_index=True)
  index += 1
